### Delivery Logistics Dataset (India – Multi-Partner)


### About Dataset
This dataset provides an extensive and realistic representation of last-mile delivery logistics across multiple regions and delivery partners in India. It contains 25,000 delivery records, each reflecting operational conditions, package characteristics, environmental factors, and delivery outcomes.

The data includes major logistics providers such as Delhivery, Blue Dart, Ekart, DHL, FedEx, Shadowfax, XpressBees, Amazon Logistics, and Ecom Express. Each record describes key delivery attributes including package type, vehicle used, delivery mode, weather condition, travel distance, package weight, and cost estimation.

Performance-related fields such as actual delivery time, expected time, delay status, and final delivery status provide insights into how real-world constraints like traffic, vehicle type, and weather influence delivery efficiency. Additionally, a delivery rating is included to reflect customer feedback, helping explore service quality patterns.

The dataset is ideal for studying logistics operations, delay analysis, delivery cost behavior, route efficiency, environmental impact, and overall supply chain performance. Since the dataset is synthetically generated, it contains no personal information and is safe for academic, analytical, or business-oriented research.

The dataset was created using a multi-stage synthetic modeling pipeline:

Delivery Setup Each record is assigned: A delivery partner A vehicle type A package type (electronics, groceries, furniture, fragile items, etc.) A delivery mode (Standard, Express, Same Day, Two Day)

Distance & Weight Generation Distance ranges from 1–300 km and package weight from 0.2–50 kg, representing urban, semi-urban, and intercity operations.

Environmental Impact Weather conditions such as Rainy, Foggy, Stormy, Hot, and Cold influence travel time using probabilistic delays.

Delivery Time Calculation Delivery time is computed using: Base travel time (depending on distance and vehicle speed) Weather delays Traffic delays

Expected Time & Delay Identification Expected delivery time varies by mode (Express, Same Day, etc.). Delay is marked as “Yes” if actual time exceeds expected time.

Delivery Status & Rating If not delayed → Delivered. If delayed → “Delayed” or “Failed” with realistic probability. Ratings follow service outcome: higher for successful deliveries, lower for failures.

Cost Generation Cost is based on: Distance Weight Delivery mode (Express and Same Day include surcharges)

Validation Data validated for: Logical consistency Range verification Clean formatting No missing values

In [1]:
#importing library
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor,GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [2]:
#ONE HOT encoding 
def onehot(df,x):
    encoder=OneHotEncoder()
    encoded=encoder.fit_transform(df[[x]]).toarray()
    encoder_df=pd.DataFrame(encoded,columns=encoder.get_feature_names_out())
    df=pd.concat([df,encoder_df],axis=1)
    df.drop([x],axis=1,inplace=True)
    return df

# Function to convert string to hours
def parse_time_to_hours(x):
    # Split by '.' to isolate fractional seconds
    parts = x.split('.')
    if len(parts) == 2:
        fractional = float("0." + parts[1])  # convert nanoseconds-like part
    else:
        fractional = 0.0
    return fractional / 3600  # convert seconds → hours

def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [3]:
#loading Dataset
df=pd.read_csv("Delivery_Logistics.csv")

In [4]:
df.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.5600


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_id          25000 non-null  float64
 1   delivery_partner     25000 non-null  object 
 2   package_type         25000 non-null  object 
 3   vehicle_type         25000 non-null  object 
 4   delivery_mode        25000 non-null  object 
 5   region               25000 non-null  object 
 6   weather_condition    25000 non-null  object 
 7   distance_km          25000 non-null  float64
 8   package_weight_kg    25000 non-null  float64
 9   delivery_time_hours  25000 non-null  object 
 10  expected_time_hours  25000 non-null  object 
 11  delayed              25000 non-null  object 
 12  delivery_status      25000 non-null  object 
 13  delivery_rating      25000 non-null  int64  
 14  delivery_cost        25000 non-null  float64
dtypes: float64(4), int64(1), object(10)


In [6]:
#checking null value
df.isnull().sum()
#observation : no null values

delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km            0
package_weight_kg      0
delivery_time_hours    0
expected_time_hours    0
delayed                0
delivery_status        0
delivery_rating        0
delivery_cost          0
dtype: int64

In [7]:

## feature engineering delivery status

# droping delivery_id , no use for training model
df.drop(['delivery_id'],axis=1,inplace=True)

# using ONE-HOT encoding and droping column "delayed" because it already can be checked by delivery_status
df=onehot(df,'delivery_status')
df.drop(['delayed'],axis=1,inplace=True)

#encoding  delivery_partner
df=onehot(df,'delivery_partner')

#encoding  "weather_condition"
df=onehot(df,"weather_condition")

#encoding package_type
df=onehot(df,"package_type")

#encoding 'vehicle_type'
df=onehot(df,'vehicle_type')

#encoding 'region'
df=onehot(df,'region')

#ordinal encoding on delivery_mode
delivery_mode_map = {
    'same day': 3,
    'express': 2,
    'two day': 1,
    'standard': 0
}
df['delivery_mode'] = df['delivery_mode'].map(delivery_mode_map)


# converting dilvery time and expected time to delay time
# Apply conversion
df['delivery_time_hours'] = df['delivery_time_hours'].apply(parse_time_to_hours)
df['expected_time_hours'] = df['expected_time_hours'].apply(parse_time_to_hours)
# Calculate delay
df['delay_time_hours'] = df['delivery_time_hours'] - df['expected_time_hours']
# Set delay_time_hours to 0 where delivery_status_delayed is 0
df.loc[df['delivery_status_delayed'] == 0, 'delay_time_hours'] = 0
# Drop originals
df.drop(['delivery_time_hours', 'expected_time_hours'], axis=1, inplace=True)




In [8]:
df.head()

,delivery_mode,distance_km,package_weight_kg,delivery_rating,delivery_cost,delivery_status_delayed,delivery_status_delivered,delivery_status_failed,delivery_partner_amazon logistics,delivery_partner_blue dart,...,vehicle_type_ev van,vehicle_type_scooter,vehicle_type_truck,vehicle_type_van,region_central,region_east,region_north,region_south,region_west,delay_time_hours
0,3,297.0,46.96,3,1632.7206,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2,89.6,47.39,5,640.1700,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,1,273.5,26.89,4,1448.1700,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,3,269.7,12.69,3,1486.5700,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1,256.7,37.02,4,1394.5600,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 44 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   delivery_mode                      25000 non-null  int64  
 1   distance_km                        25000 non-null  float64
 2   package_weight_kg                  25000 non-null  float64
 3   delivery_rating                    25000 non-null  int64  
 4   delivery_cost                      25000 non-null  float64
 5   delivery_status_delayed            25000 non-null  float64
 6   delivery_status_delivered          25000 non-null  float64
 7   delivery_status_failed             25000 non-null  float64
 8   delivery_partner_amazon logistics  25000 non-null  float64
 9   delivery_partner_blue dart         25000 non-null  float64
 10  delivery_partner_delhivery         25000 non-null  float64
 11  delivery_partner_dhl               25000 non-null  flo

In [10]:
X=df.drop(['delivery_cost'],axis=1)
y=df['delivery_cost']

In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)


In [12]:

# Identify numeric features (continuous variables to scale)
numeric_features = ['distance_km', 'package_weight_kg', 'delay_time_hours']

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ],
    remainder='passthrough'  # keep categorical one-hot features as they are
)

# Apply preprocessing
X_train = preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

# Convert back to DataFrame (optional, for inspection)
X_train = pd.DataFrame(X_train, columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(X_test, columns=preprocessor.get_feature_names_out())

In [13]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "SVR":SVR(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoostRegressor":AdaBoostRegressor(),
    "GradientBoostingRegressor":GradientBoostingRegressor()
   
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 14.3250
- Mean Absolute Error: 12.5765
- R2 Score: 0.9989
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 14.3194
- Mean Absolute Error: 12.5629
- R2 Score: 0.9989


Lasso
Model performance for Training set
- Root Mean Squared Error: 14.4484
- Mean Absolute Error: 12.7048
- R2 Score: 0.9989
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 14.4138
- Mean Absolute Error: 12.6534
- R2 Score: 0.9989


Ridge
Model performance for Training set
- Root Mean Squared Error: 14.3250
- Mean Absolute Error: 12.5775
- R2 Score: 0.9989
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 14.3195
- Mean Absolute Error: 12.5638
- R2 Score: 0.9989


SVR
Model performance for Training set
- Root Mean Squared Error: 64.1378
- Mean Absolute Error: 39.9178
- R2 Score: 0.9784
---------------------------